# 02 — Warehouse access and training datasets

Notebook 01 built the **warehouse**. This notebook turns it into a **versioned training dataset** the rest of the pipeline (preprocessing, models, training, evaluation, inference) consumes.

## The architecture in one picture

A *dataset* is two things:

1. A **recipe** — `FeatureSelection` + date range + locations + QC filter. Pure Python config, lives in this repo, in git.
2. A **materialisation** — the parquet payload produced by applying the recipe to the warehouse at a specific point in time, plus a `manifest.json` capturing the warehouse state, the git SHA, and a SHA-256 of the parquet for tamper-detection.

The two-layer split is what lets us answer questions like:

* "What if we ingest more ground truth?" → same recipe, new materialisation, bumped version.
* "What if we add a new variable?" → new recipe, new artifact family.
* "What if we want fewer variables for an ablation?" → derived recipe, separate materialisation.
* "What if upstream data was wrong?" → fix in warehouse, re-run recipe, new materialisation.

## Where snapshots live

Each snapshot is a directory with `dataset.parquet` + `manifest.json`. Locally we write to `data/training_snapshots/<name>_<version>/`; from there we upload to **W&B Artifacts** of type `training_dataset`. Every training run consumes a specific artifact version, so the lineage graph in W&B answers "which model used which dataset" automatically.


## 0 — Setup


In [12]:
%load_ext autoreload
%autoreload 2

import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# Make the in-repo `src/` importable when running from `notebooks/`.
src_path = (Path.cwd() / "../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load .env so WANDB_API_KEY (if present) is picked up automatically.
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

print("Python", sys.version.split()[0])


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python 3.12.1


In [13]:
from susse.warehouse_ops.io import BigQueryClient, TableRefs

bq = BigQueryClient()
tables = TableRefs(config=bq.config)
print(f"Connected to {bq.config.project_id}.{bq.config.dataset}.")


Connected to solar-irradiation-estimation.solar_warehouse.


## 1 — Define the recipe

`FeatureSelection` enumerates which variables to materialise per source, plus the QC filter and which satellite-irradiance series to include. Validation against the warehouse catalog runs at construction time, so a typo here surfaces as a `ValueError` *before* you spend a query on it.

For **v1** we pick a deliberately small recipe: ground GHI as the target, NASA + CAMS satellite GHI as the primary features, and three NASA atmospheric covariates. We'll bump to v2 once B8 finishes adding MERRA-2 coverage to all stations, and v3 once C8 lands MODIS.


In [14]:
from susse.datasets import FeatureSelection
from susse.warehouse_ops.population.types import Source

# v1 recipe: aerosols + water vapour + cloudiness + NASA's pre-computed
# clearness_index (already a kt feature) on the NASA side, and CAMS
# `ghi_clear` on the CAMS side so NB 03 can derive `kt_cams = sat_ghi_cams
# / cams_ghi_clear`. Keeping NB 02 and NB 03 recipes in lockstep is the
# whole point of the recipe-vs-materialisation split: bumping either
# one means re-deriving the snapshot.
selection_v1 = FeatureSelection(
    nasa_variable_ids=(
        "aod_550", "precipitable_water", "cloud_amount", "clearness_index",
    ),
    cams_variable_ids=("ghi_clear",),
    merra_variable_ids=(),
    modis_variable_ids=(),
    include_satellite_irradiance=(Source.NASA_POWER, Source.CAMS),
    qc_levels=("pass",),
)
print("Recipe columns:", ("y_ghi_kwh_m2_day",) + selection_v1.aux_columns)
print("Recipe is empty?", selection_v1.is_empty)

Recipe columns: ('y_ghi_kwh_m2_day', 'nasa_aod_550', 'nasa_precipitable_water', 'nasa_cloud_amount', 'nasa_clearness_index', 'cams_ghi_clear')
Recipe is empty? False


## 2 — Build training pairs

`FeatureService.build_training_pairs` runs the SQL, applies the QC filter, and pivots each long-format aux table into source-prefixed wide columns. Note the **source prefix** on aux columns (`nasa_aod_550`, not `aod_550`) — when MERRA-2 lands its own `aod_550` analysis variable in v2, the names won't collide.


In [15]:
from susse.datasets import FeatureService

fs = FeatureService(bq=bq, tables=tables)
training_df = fs.build_training_pairs(
    selection=selection_v1,
    date_start=date(2024, 1, 1),
    date_end=date(2024, 6, 30),
)
print(f"Loaded {len(training_df):,} training rows × {len(training_df.columns)} columns.")
print("Columns:", list(training_df.columns))
training_df.head(3)


/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 3,710 training rows × 14 columns.
Columns: ['date', 'location', 'lat', 'lon', 'geohash5', 'y_ghi_kwh_m2_day', 'qc_level', 'sat_ghi_nasa_kwh_m2_day', 'sat_ghi_cams_kwh_m2_day', 'nasa_aod_550', 'nasa_precipitable_water', 'nasa_cloud_amount', 'nasa_clearness_index', 'cams_ghi_clear']


,date,location,lat,lon,geohash5,y_ghi_kwh_m2_day,qc_level,sat_ghi_nasa_kwh_m2_day,sat_ghi_cams_kwh_m2_day,nasa_aod_550,nasa_precipitable_water,nasa_cloud_amount,nasa_clearness_index,cams_ghi_clear
0,2024-01-01,egypt_location1,29.887561,32.460447,str44,3.128732,pass,3.7332,3.712784,0.28,1.60,6.78,0.66,3.869308
1,2024-01-01,ghana_location1,5.645759,-0.105223,ecpbj,4.246611,pass,5.3671,5.420120,0.57,4.28,8.20,0.58,5.679153
2,2024-01-01,ghana_location2,5.645686,0.008678,s1000,4.179786,pass,4.8605,5.340148,0.57,4.23,5.13,0.53,5.567244


A quick sanity check: per-station coverage of the target column. Stations with zero rows here either have no QC-passed ground data in the requested window, or weren't ingested yet.


In [16]:
training_df.groupby("location").size().sort_values(ascending=False).head(10)


location
kenya_location13        344
kenya_location10        322
kenya_location5         289
kenya_location11        233
madagascar_location2    182
kenya_location8         182
egypt_location1         182
ghana_location1         181
kenya_location3         180
nigeria_location2       180
dtype: int64

## 3 — Build inference features

The same recipe drives inference. Given a `(lat, lon, date)`, `FeatureService.build_inference_features` returns a single-row frame with the satellite GHI estimates and the same auxiliary columns — but no target. This is what the portal will call in production.


In [17]:
inference_today = fs.build_inference_features(
    selection=selection_v1,
    target_date=date(2024, 6, 15),
    lat=0.5179,
    lon=32.4715,  # central Uganda grid point
)
inference_today


/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date,geohash5,sat_ghi_nasa_kwh_m2_day,sat_ghi_cams_kwh_m2_day,nasa_aod_550,nasa_precipitable_water,nasa_cloud_amount,nasa_clearness_index,cams_ghi_clear,lat,lon
0,2024-06-15,s8p4f,5.21,4.730357,0.4,3.42,34.93,0.56,6.447131,0.5179,32.4715


## 4 — Materialise as a snapshot (local)

`build_and_write_snapshot` queries the warehouse, captures provenance (git SHA, susse version, per-source-table modification timestamps), writes `dataset.parquet` + `manifest.json` into the destination directory, and stamps the manifest with the parquet's SHA-256 hash.


In [18]:
from susse.datasets import build_and_write_snapshot

SNAPSHOT_ROOT = (Path.cwd() / "../data/training_snapshots").resolve()
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)

dataset_name = "susse_training_demo"
dataset_version = "v1-2024H1"
snapshot_dir = SNAPSHOT_ROOT / f"{dataset_name}_{dataset_version}"

snapshot = build_and_write_snapshot(
    feature_service=fs,
    selection=selection_v1,
    date_start=date(2024, 1, 1),
    date_end=date(2024, 6, 30),
    locations=None,
    name=dataset_name,
    version=dataset_version,
    dest=snapshot_dir,
)
print(f"Wrote {snapshot.manifest.n_rows} rows × {snapshot.manifest.n_cols} cols")
print(f"  to {snapshot_dir}")
print(f"  hash {snapshot.manifest.content_hash[:16]}...")


/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Wrote 3710 rows × 14 cols
  to /home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/data/training_snapshots/susse_training_demo_v1-2024H1
  hash 8e5c9923025bf7d7...


/home/jan/Dropbox/personal_projects/Irradiation_project/Solar_irradiation/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 5 — Reload from local + verify

`load_snapshot` reads the parquet, reconstructs the manifest, and validates the parquet's SHA-256 against the manifest. Tampering or corruption raises a clear error before training silently consumes broken data.


In [19]:
from susse.datasets import load_snapshot

reloaded = load_snapshot(snapshot_dir)
pd.testing.assert_frame_equal(reloaded.df, snapshot.df)
print(f"Roundtrip OK. {reloaded.manifest.name} {reloaded.manifest.version}")


Roundtrip OK. susse_training_demo v1-2024H1


## 6 — Inspect the manifest

The manifest is the single source of provenance truth for the snapshot. Every field is JSON-serialisable and survives the W&B upload as `manifest.json` *inside* the artifact (a subset is also mirrored into the artifact's `metadata` for the W&B UI).


In [20]:
import json
print(json.dumps(reloaded.manifest.to_dict(), indent=2)[:2200])


{
  "name": "susse_training_demo",
  "version": "v1-2024H1",
  "created_at_utc": "2026-05-08T23:35:32.856445+00:00",
  "susse_version": "0.1.dev324+g28cd6fce8.d20260508",
  "git_sha": "80d9a0928d0b11ce4b63b780e2398fc842808d61",
  "feature_selection": {
    "nasa_variable_ids": [
      "aod_550",
      "precipitable_water",
      "cloud_amount",
      "clearness_index"
    ],
    "cams_variable_ids": [
      "ghi_clear"
    ],
    "merra_variable_ids": [],
    "modis_variable_ids": [],
    "include_satellite_irradiance": [
      "NASA",
      "CAMS"
    ],
    "qc_levels": [
      "pass"
    ]
  },
  "date_start": "2024-01-01",
  "date_end": "2024-06-30",
  "location_filter": null,
  "warehouse_project": "solar-irradiation-estimation",
  "warehouse_dataset": "solar_warehouse",
  "warehouse_table_mods": {
    "cams_daily_vars_long": "2026-05-08T12:57:27.961000+00:00",
    "ground_measurements": "2025-10-27T12:11:17.877000+00:00",
    "irradiance_daily": "2026-05-08T12:56:31.728000+00:00"

## 7 — Log to W&B as an artifact

Set `LOG_TO_WANDB = True` to actually upload. Default is `False` so the notebook is safe to re-execute without hitting the W&B API on every cell run.

The upload creates a `dataset_build` run that records the recipe + warehouse state in its config, then attaches the snapshot directory as an artifact of type `training_dataset`. W&B assigns its own `v0`, `v1`, ... versioning on top of our human-readable `manifest.version`.


In [21]:
LOG_TO_WANDB = False  # flip to True to upload
WANDB_PROJECT = "susse"
WANDB_ENTITY = None  # None = your default entity from `wandb login`

if LOG_TO_WANDB:
    from susse.datasets import log_dataset_artifact
    artifact_ref = log_dataset_artifact(
        snapshot_dir,
        artifact_name=dataset_name,
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        aliases=(dataset_version,),
    )
    print(f"Logged: {artifact_ref}")
else:
    artifact_ref = None
    print("LOG_TO_WANDB=False — skipping upload. Set True to push.")


LOG_TO_WANDB=False — skipping upload. Set True to push.


## 8 — Reload from W&B + verify

`use_dataset_artifact` downloads the artifact contents and runs the same content-hash validation as the local loader. Inside a training run (NB 05), pass `run=` so the consumption shows up in the lineage graph.


In [22]:
DOWNLOAD_FROM_WANDB = False  # flip to True after a successful upload above

if DOWNLOAD_FROM_WANDB and artifact_ref is not None:
    from susse.datasets import use_dataset_artifact
    download_dest = SNAPSHOT_ROOT / f"_wandb_download_{dataset_name}_{dataset_version}"
    redownloaded = use_dataset_artifact(artifact_ref, download_dest)
    pd.testing.assert_frame_equal(redownloaded.df, reloaded.df)
    print(f"W&B roundtrip OK. {redownloaded.manifest.name} {redownloaded.manifest.version}")
else:
    print(
        "DOWNLOAD_FROM_WANDB=False or no upload yet — skipping. "
        "Run cell 7 with LOG_TO_WANDB=True first."
    )


DOWNLOAD_FROM_WANDB=False or no upload yet — skipping. Run cell 7 with LOG_TO_WANDB=True first.


## What's next

Notebook 03 (`03_preprocessing.ipynb`) takes a `TrainingDataset` and applies preprocessing: clear-sky-index features (`kt = GHI / GHI_clear`), missing-value handling, train/val splits. The `Preprocessor` it produces will be stored alongside the model so inference uses identical transformations.

The **versioning discipline** to follow as the warehouse grows:

| Change | Action |
|---|---|
| Bug fix to one variable's curation | Apply migration → re-run recipe → bump materialisation version (`v1.1`) |
| New ground stations ingested | Re-run recipe → bump version (`v2`) |
| Add a new aux variable | New `FeatureSelection` → new artifact family |
| Smaller subset for an ablation | Derive a child recipe → separate artifact name |

The recipe itself never gets edited in place — that would silently change what `v1` means. New variants get new names.
